# 01 - Harmonize the state liming recommendations

Read `data/lime-recs.xlsx` (one sheet per state) and write `data/recs_harmonized.csv`. We use:

- dose as **kg CCE ha⁻¹** = `dose_native` × unit factor × `cce_factor` `cce_assumed` is 1, and as **mmol_c kg⁻¹ soil** using `mixdepth_cm` and bulk density (assumed 1.3 g cm-3 where blank, `bd_assumed`);
- the same dose as **t CO₂-eq ha⁻¹** (`co2_eq_t_ha`), the CO₂ that the released cations could charge-balance as bicarbonate (each Ca²⁺ balances 2 HCO₃⁻, so 2 × 44.01 / 100.09 = 0.88 t CO₂ per t CCE);
- **DpH** = `ph_target − ph_initial` where an initial pH is given. Otherwise DpH comes from differences between targets in the same soil, anchored at the soil's lowest target; the dose is also taken as the difference over this anchor dose.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

DATA = Path("../data")
LB_ACRE = {"lb_acre": 1.0, "short_ton_acre": 2000.0, "lb_1000sqft": 43.56}   # unit -> lb per acre
KG_HA_PER_LB_ACRE = 1.12085
BD_DEFAULT = 1.3            # g cm-3, used where a source gives no bulk density
MG_PER_MMOLC = 50.04        # CaCO3: 100.09 g mol-1 / 2 eq
CO2_PER_CCE = 2 * 44.01 / 100.09   # t CO2 per t CaCO3, as HCO3- charge-balanced by Ca2+ (2 eq mol-1)
COLS = ["dataset", "kind", "state", "source_name", "soil_id", "ph0", "ph_target", "dpH", "dpH_basis",
        "dose_native", "dose_units", "basis_kind", "cce_factor", "cce_assumed", "mixdepth_cm", "bd", "bd_assumed",
        "dose_kgCCE_ha", "co2_eq_t_ha", "dose_mmolc_kg", "dose_dpH_kgCCE_ha", "dose_dpH_mmolc_kg",
        "above_max_rate", "flag", "ph_note", "note", "source_url"]
soil_kg_ha = lambda depth_cm, bd: depth_cm * bd * 1e5     # mass of the mixing layer

In [2]:
XLSX = DATA / "lime-recs.xlsx"

def read_sheet(name):
    raw = pd.read_excel(XLSX, sheet_name=name, header=None)
    hdr = raw.index[(raw == "source_name").any(axis=1)][0]          # header row (first row is blank)
    df = pd.read_excel(XLSX, sheet_name=name, header=hdr)
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].dropna(how="all").assign(state=name)

raw = pd.concat([read_sheet(s) for s in pd.ExcelFile(XLSX).sheet_names], ignore_index=True)
print(raw.shape, "rows x cols from", raw.state.nunique(), "sheets")

(3589, 19) rows x cols from 38 sheets


States with no usable rows (placeholder sheets):

In [3]:
raw[raw.dose_native.isna()][["state", "note"]].fillna("")

,state,note
241,CT,no CT-specific lime rate recs found
734,HI,Liming recommendations are displayed graphical...
777,ID,Data presented are measured pH responses rathe...
988,LA,Limited liming information and couldn't find u...
989,MA,mass appears to have just a worked example rat...
1812,MS,could not find a liming rec table. omitting fo...
2069,ND,"Rule of thumb but no direct calibration, omitt..."
2070,NE,Omitting for now. Only has a single target pH ...
3167,SD,omitting for now because no clear target or in...
3168,TX,"no clear target pH given, omitting for now."


In [4]:
df = raw.dropna(subset=["dose_native", "ph_target"]).copy()

df["soil_id"] = df.soil.astype(str)
df["ph_initial"], df["ph_target"] = df.ph_initial.round(3), df.ph_target.round(3)   # strip float noise from Excel

# dose on a common basis
df["cce_assumed"] = df.cce_factor.isna()
df["bd_assumed"] = df.bulk_density_g_cm3.isna()
df["bd"] = df.bulk_density_g_cm3.fillna(BD_DEFAULT)
lb_acre = df.dose_native * df.dose_units.map(LB_ACRE)
df["dose_kgCCE_ha"] = lb_acre * KG_HA_PER_LB_ACRE * df.cce_factor.fillna(1.0)
df["co2_eq_t_ha"] = df.dose_kgCCE_ha * CO2_PER_CCE / 1000
df["dose_mmolc_kg"] = df.dose_kgCCE_ha * 1e6 / soil_kg_ha(df.mixdepth_cm, df.bd) / MG_PER_MMOLC
df["above_max_rate"] = lb_acre > df.max_rate * df.max_rate_units.map(LB_ACRE)

# delta pH and the dose that goes with it
grp = ["state", "source_name", "soil_id"]
has0 = df.ph_initial.notna()
tmin = df.groupby(grp).ph_target.transform("min")
df["ph0"] = df.ph_initial.where(has0, tmin)
df["dpH"] = df.ph_target - df.ph0
df["dpH_basis"] = np.where(has0, "initial_pH", "target_difference")
for u in ["kgCCE_ha", "mmolc_kg"]:
    at_anchor = df[f"dose_{u}"].where(df.ph_target.eq(tmin)).groupby([df[c] for c in grp]).transform("first")
    df[f"dose_dpH_{u}"] = df[f"dose_{u}"].where(has0, df[f"dose_{u}"] - at_anchor)

df["flag"] = np.select([df.dpH <= 0, df.dose_dpH_mmolc_kg <= 0], ["dpH<=0", "dose<=0"], "")
df["dataset"], df["kind"] = "extension", "recommendation"
recs = df[COLS].reset_index(drop=True)
recs.to_csv(DATA / "recs_harmonized.csv", index=False)
print(len(recs), "rows,", recs.state.nunique(), "states;", recs.flag.value_counts().to_dict())

3578 rows, 27 states; {'': 3171, 'dpH<=0': 276, 'dose<=0': 131}


In [5]:
(recs.assign(usable=recs.flag.eq(""))
     .groupby("state")
     .agg(rows=("dpH", "size"), usable=("usable", "sum"), soils=("soil_id", "nunique"),
          basis=("dpH_basis", "first"), units=("dose_units", "first"), cce=("cce_factor", "first"),
          median_mmolc_per_pH=("dose_dpH_mmolc_kg", lambda v: (v / recs.loc[v.index, "dpH"])[recs.loc[v.index, "flag"].eq("")].median().round(1))))

,rows,usable,soils,basis,units,cce,median_mmolc_per_pH
state,,,,,,,
AK,45,30,15,target_difference,short_ton_acre,1.000,43.8
AL,52,43,1,initial_pH,lb_acre,0.670,12.6
AR,24,24,8,initial_pH,lb_acre,0.915,91.5
CA,96,88,6,initial_pH,lb_acre,1.000,15.3
CO,24,24,6,initial_pH,lb_acre,1.000,35.8
DE,243,229,3,initial_pH,short_ton_acre,0.670,51.6
GA,249,249,3,initial_pH,lb_acre,1.000,47.2
IA,42,26,14,target_difference,lb_acre,1.000,66.6
IL,35,32,10,initial_pH,short_ton_acre,0.900,72.4
